# **Mini Project: Smart Urban Mobility Analytics Platform (SUMAP)**
## **NoSQL Data Ingestion & Analytics Pipeline (Neo4j)**
### **Course:** NOSQL DATABASES (MCST1133)

### **Group:** 5 (UC1)

### **Team Members**:

CHENG YUN - MCS241014

SIVARAJAN A/L S.ESVARAN - MCS241051

LI HONGLIN - MCS241031

CUI ZHIWEN - MCS241040




In [ ]:
#install neo4j and tdqm
!pip install neo4j
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 6.6 MB/s eta 0:00:00


#Import Libraries

In [ ]:
import pandas as pd
from neo4j import GraphDatabase
from tqdm import tqdm

#Dataset information analysing

In [ ]:
#passenger entity
path='/content/passengers.csv'
df_passengers = pd.read_csv(path)
print(df_passengers.head())
print(df_passengers.info())

                           passenger_id             full_name  age  \
0  57b94dfb-af82-470c-b964-4e4f2aefcf3b      William Jennings   22   
1  80da2ed7-880b-4b0e-88ab-96ace0b4659a  Lesley Wilson-Newman   16   
2  d6521e08-c2f4-4c10-a869-707cbdcbfffc         Abdul Hopkins   28   
3  68aa1236-50de-4f43-8de5-c35ab0117b0a      Mrs Lauren Green   45   
4  5827bc1d-93e0-4235-8e80-d10bfd0ee0b6           Tina Foster   21   

              gender                  email          phone home_zone  \
0  Prefer not to say  user0001@gmail.com.my  +6013-2615622    Zone B   
1  Prefer not to say     user0002@yahoo.com  +6014-9088689    Zone A   
2               Male     user0003@gmail.com  +6011-9766777    Zone A   
3             Female     user0004@gmail.com  +6017-9094465    Zone C   
4  Prefer not to say  user0005@gmail.com.my  +6016-1873301    Zone F   

  work_zone commuter_type        registered_at        preferred_mode  \
0    Zone B       Student  2024-06-12T00:10:57                   LRT   
1 

In [ ]:
#Journey entity
path='/content/journeys.csv'
df_journeys = pd.read_csv(path)
print(df_journeys.head())
print(df_journeys.info())
print(df_journeys.sort_values(by=['journey_date']))

                             journey_id                          passenger_id  \
0  f9648a0f-2d04-40a7-945d-eec7019dc12d  7aededf4-6f30-4743-820e-2e6abcc8509b   
1  acac1b18-be88-44ab-ba27-a8f803125cdd  366b3575-d07a-4518-912c-f322735e8153   
2  f22fdeb8-97ca-4ad1-a6aa-78092c3b4e9c  c138a98b-5c88-4a6e-a551-da7a7011a2be   
3  0351d800-7d91-4bcc-aa95-9ce237663e35  141fb001-f7bc-4c14-9241-9671da2fc99b   
4  f7294c21-208b-4725-a6e0-05dd31c2bddb  397e241d-fe08-4540-af7e-1940f5687574   

          journey_date origin_stop_id destination_stop_id transport_mode  \
0  2022-02-25T21:10:40       STOP-045            STOP-048     Multimodal   
1  2022-08-17T09:50:57       STOP-051            STOP-022     Multimodal   
2  2022-02-06T20:01:50       STOP-079            STOP-064     Multimodal   
3  2024-09-02T06:05:12       STOP-006            STOP-003     Multimodal   
4  2023-09-03T17:01:16       STOP-060            STOP-041     Multimodal   

   total_fare_myr fare_category payment_method  duration

In [ ]:
#Route entity
path='/content/routes.csv'
df_routes = pd.read_csv(path)
print(df_routes.head())
print(df_routes.info())

  route_id                       route_name mode         operator  \
0   BRT-01  City Centre – Northgate Express  BRT  NusaBRT Sdn Bhd   
1   BRT-02         University Loop – Zone C  BRT  NusaBRT Sdn Bhd   
2   BRT-03    Airport – Central Interchange  BRT  NusaBRT Sdn Bhd   
3   BRT-04      Southgate – Industrial Park  BRT  NusaBRT Sdn Bhd   
4   BRT-05      West Market – East Terminal  BRT  NusaBRT Sdn Bhd   

   total_stops  distance_km  frequency_mins  is_express start_time end_time  \
0           12         18.5               7        True      05:30    23:30   
1           18         22.1              10       False      05:45    23:00   
2            8         34.0              12        True      05:00    01:00   
3           14         19.8              10       False      05:30    22:30   
4           20         26.3               8       False      06:00    22:00   

   avg_daily_passengers  status  
0                 38000  Active  
1                 22000  Active  
2       

#Connect Neo4j

In [ ]:
URI = "neo4j+s://be0a3f61.databases.neo4j.io"

USERNAME = "be0a3f61"
PASSWORD = "wcEdfl7sSqjEjtNsdGiLIIuWzqZ7m2Q3OwVz47a3N24"

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected Successfully")

Connected Successfully


#Clear Database

In [ ]:
with driver.session() as session:

    session.run("""
    MATCH (n)
    DETACH DELETE n
    """)

print("Database Cleared")

Database Cleared


#Create Constraint

In [ ]:
constraints = [

"""
CREATE CONSTRAINT passenger_id_unique
IF NOT EXISTS
FOR (p:Passenger)
REQUIRE p.passenger_id IS UNIQUE
""",

"""
CREATE CONSTRAINT journey_id_unique
IF NOT EXISTS
FOR (j:Journey)
REQUIRE j.journey_id IS UNIQUE
""",

"""
CREATE CONSTRAINT route_id_unique
IF NOT EXISTS
FOR (r:Route)
REQUIRE r.route_id IS UNIQUE
""",

"""
CREATE CONSTRAINT stop_id_unique
IF NOT EXISTS
FOR (s:Stop)
REQUIRE s.stop_id IS UNIQUE
""",

"""
CREATE CONSTRAINT vehicle_id_unique
IF NOT EXISTS
FOR (v:Vehicle)
REQUIRE v.vehicle_id IS UNIQUE
""",

"""
CREATE CONSTRAINT feedback_id_unique
IF NOT EXISTS
FOR (f:Feedback)
REQUIRE f.feedback_id IS UNIQUE
"""
]

with driver.session() as session:

    for c in constraints:
        session.run(c)

print("Constraints Created")

Constraints Created


#Read and Loan CSV file

In [ ]:
passengers = pd.read_csv("/content/passengers.csv")
journeys = pd.read_csv("/content/journeys.csv")
routes = pd.read_csv("/content/routes.csv")
stops = pd.read_csv("/content/stops.csv")
vehicles = pd.read_csv("/content/vehicles.csv")
feedback = pd.read_csv("/content/feedback.csv")

print("Files Loaded Successfully")

Files Loaded Successfully


#Show dataset size

In [ ]:
print("Passengers :", len(passengers))
print("Journeys   :", len(journeys))
print("Routes     :", len(routes))
print("Stops      :", len(stops))
print("Vehicles   :", len(vehicles))
print("Feedback   :", len(feedback))

Passengers : 500
Journeys   : 3000
Routes     : 21
Stops      : 120
Vehicles   : 250
Feedback   : 300


#Import Passenger Nodes

In [ ]:
query = """
MERGE (p:Passenger {
    passenger_id:$passenger_id
})
SET p += $props
"""

with driver.session() as session:

    for _, row in tqdm(
        passengers.iterrows(),
        total=len(passengers)
    ):

        props = row.to_dict()

        props["preferred_mode"] = \
            str(props["preferred_mode"]).split("|")

        session.run(
            query,
            passenger_id=row["passenger_id"],
            props=props
        )

print("Passengers Imported")

100%|██████████| 500/500 [01:58<00:00,  4.23it/s]

Passengers Imported


#Import Journary Node

In [ ]:
query = """
MERGE (j:Journey {
    journey_id:$journey_id
})
SET j += $props
"""

with driver.session() as session:

    for _, row in tqdm(
        journeys.iterrows(),
        total=len(journeys)
    ):

        session.run(
            query,
            journey_id=row["journey_id"],
            props=row.to_dict()
        )

print("Journeys Imported")

100%|██████████| 3000/3000 [11:52<00:00,  4.21it/s]

Journeys Imported


#Import Route Node

In [ ]:
query = """
MERGE (r:Route {
    route_id:$route_id
})
SET r += $props
"""

with driver.session() as session:

    for _, row in tqdm(
        routes.iterrows(),
        total=len(routes)
    ):

        session.run(
            query,
            route_id=row["route_id"],
            props=row.to_dict()
        )

print("Routes Imported")

100%|██████████| 21/21 [00:05<00:00,  4.14it/s]

Routes Imported


#Import Stop Node

In [ ]:
query = """
MERGE (s:Stop {
    stop_id:$stop_id
})
SET s += $props
"""

with driver.session() as session:

    for _, row in tqdm(
        stops.iterrows(),
        total=len(stops)
    ):

        props = row.to_dict()

        props["facilities"] = \
            str(props["facilities"]).split("|")

        session.run(
            query,
            stop_id=row["stop_id"],
            props=props
        )

print("Stops Imported")

100%|██████████| 120/120 [00:30<00:00,  3.90it/s]

Stops Imported


#Import Vehicle Node

In [ ]:
query = """
MERGE (v:Vehicle {
    vehicle_id:$vehicle_id
})
SET v += $props
"""

with driver.session() as session:

    for _, row in tqdm(
        vehicles.iterrows(),
        total=len(vehicles)
    ):

        session.run(
            query,
            vehicle_id=row["vehicle_id"],
            props=row.to_dict()
        )

print("Vehicles Imported")

100%|██████████| 250/250 [01:01<00:00,  4.06it/s]

Vehicles Imported


#Import Feedback Node

In [ ]:
query = """
MERGE (f:Feedback {
    feedback_id:$feedback_id
})
SET f += $props
"""

with driver.session() as session:

    for _, row in tqdm(
        feedback.iterrows(),
        total=len(feedback)
    ):

        props = row.to_dict()

        props["tags"] = \
            str(props["tags"]).split("|")

        session.run(
            query,
            feedback_id=row["feedback_id"],
            props=props
        )

print("Feedback Imported")

100%|██████████| 300/300 [01:12<00:00,  4.12it/s]

Feedback Imported


#Verifiy Node Count

In [ ]:
with driver.session() as session:

    result = session.run("""

    MATCH (n)

    RETURN labels(n)[0] AS Label,
           count(*) AS Count

    """)

    for row in result:
        print(row)

<Record Label='Journey' Count=3000>
<Record Label='Route' Count=21>
<Record Label='Stop' Count=120>
<Record Label='Vehicle' Count=250>
<Record Label='Feedback' Count=300>
<Record Label='Passenger' Count=500>


#Sample Passenger Node

In [ ]:
with driver.session() as session:

    result = session.run("""

    MATCH (p:Passenger)

    RETURN p

    LIMIT 3

    """)

    for row in result:
        print(row)

<Record p=<Node element_id='4:cc9c5f66-9ea1-4044-a198-f5038bb10831:4191' labels=frozenset({'Passenger'}) properties={'is_active': True, 'gender': 'Prefer not to say', 'registered_at': '2024-06-12T00:10:57', 'loyalty_tier': 'Bronze', 'total_journeys': 49, 'passenger_id': '57b94dfb-af82-470c-b964-4e4f2aefcf3b', 'home_zone': 'Zone B', 'full_name': 'William Jennings', 'commuter_type': 'Student', 'phone': '+6013-2615622', 'preferred_mode': ['LRT'], 'work_zone': 'Zone B', 'age': 22, 'email': 'user0001@gmail.com.my'}>>
<Record p=<Node element_id='4:cc9c5f66-9ea1-4044-a198-f5038bb10831:4192' labels=frozenset({'Passenger'}) properties={'is_active': True, 'gender': 'Prefer not to say', 'registered_at': '2024-07-04T01:58:58', 'loyalty_tier': 'Bronze', 'total_journeys': 69, 'passenger_id': '80da2ed7-880b-4b0e-88ab-96ace0b4659a', 'home_zone': 'Zone A', 'full_name': 'Lesley Wilson-Newman', 'commuter_type': 'Student', 'phone': '+6014-9088689', 'preferred_mode': ['Scooter', 'Bike', 'BRT'], 'work_zone'

#Sample Route Node

In [ ]:
with driver.session() as session:

    result = session.run("""

    MATCH (r:Route)

    RETURN r

    LIMIT 3

    """)

    for row in result:
        print(row)

<Record r=<Node element_id='4:cc9c5f66-9ea1-4044-a198-f5038bb10831:2498' labels=frozenset({'Route'}) properties={'mode': 'BRT', 'start_time': '05:30', 'route_id': 'BRT-01', 'distance_km': 18.5, 'total_stops': 12, 'route_name': 'City Centre – Northgate Express', 'frequency_mins': 7, 'end_time': '23:30', 'is_express': True, 'operator': 'NusaBRT Sdn Bhd', 'status': 'Active', 'avg_daily_passengers': 38000}>>
<Record r=<Node element_id='4:cc9c5f66-9ea1-4044-a198-f5038bb10831:2499' labels=frozenset({'Route'}) properties={'mode': 'BRT', 'start_time': '05:45', 'route_id': 'BRT-02', 'distance_km': 22.1, 'total_stops': 18, 'route_name': 'University Loop – Zone C', 'frequency_mins': 10, 'end_time': '23:00', 'is_express': False, 'operator': 'NusaBRT Sdn Bhd', 'status': 'Active', 'avg_daily_passengers': 22000}>>
<Record r=<Node element_id='4:cc9c5f66-9ea1-4044-a198-f5038bb10831:2500' labels=frozenset({'Route'}) properties={'mode': 'BRT', 'start_time': '05:00', 'route_id': 'BRT-03', 'distance_km': 3

#Sample Vehicle Node

In [ ]:
with driver.session() as session:

    result = session.run("""

    MATCH (v:Vehicle)

    RETURN v

    LIMIT 3

    """)

    for row in result:
        print(row)

<Record v=<Node element_id='4:cc9c5f66-9ea1-4044-a198-f5038bb10831:2639' labels=frozenset({'Vehicle'}) properties={'odometer_km': 359752.6, 'tel_engine_temp_c': 77.6, 'tel_pantograph_voltage_v': nan, 'tel_door_status': nan, 'tel_last_docked_stop': nan, 'tel_lock_status': nan, 'tel_traction_mode': nan, 'last_service_date': '2024-08-20', 'tel_last_gps_ping': '2024-11-28T23:19:31', 'tel_signal_block_id': nan, 'capacity': 100, 'mode': 'BRT', 'tel_avg_speed_kmh': 42.2, 'tel_is_damaged': nan, 'tel_battery_pct': 97.0, 'manufacture_year': 2015, 'assigned_route_id': 'BRT-09', 'current_status': 'In Service', 'model': 'Yutong E12', 'tel_ac_status': 'On', 'fuel_type': 'Electric', 'vehicle_id': 'VH-BRT-001'}>>
<Record v=<Node element_id='4:cc9c5f66-9ea1-4044-a198-f5038bb10831:2640' labels=frozenset({'Vehicle'}) properties={'odometer_km': 6321.9, 'tel_engine_temp_c': 95.5, 'tel_pantograph_voltage_v': nan, 'tel_door_status': nan, 'tel_last_docked_stop': nan, 'tel_lock_status': nan, 'tel_traction_mode

#Load CSV

In [ ]:
journey_segments = pd.read_csv("/content/journey_segments.csv")
stop_routes = pd.read_csv("/content/stop_routes.csv")
stop_connections = pd.read_csv("/content/stop_connections.csv")

print("Relationship files loaded successfully")

Relationship files loaded successfully


#Create Route-Stop Relationship

In [ ]:
with driver.session() as session:

    for _, row in stop_routes.iterrows():

        session.run("""

        MATCH (r:Route {
            route_id:$route_id
        })

        MATCH (s:Stop {
            stop_id:$stop_id
        })

        MERGE (r)-[:SERVES]->(s)

        """,

        route_id=row["route_id"],
        stop_id=row["stop_id"])

print("SERVES relationships created")

SERVES relationships created


#Create Stop-Stop Relationship

In [ ]:
with driver.session() as session:

    for _, row in stop_connections.iterrows():

        session.run("""

        MATCH (a:Stop {
            stop_id:$from_stop
        })

        MATCH (b:Stop {
            stop_id:$to_stop
        })

        MERGE (a)-[:CONNECTED_TO {

            via_mode:$via_mode,
            avg_transfer_walk_mins:$walk_mins,
            peak_hour_only:$peak_hour

        }]->(b)

        """,

        from_stop=row["from_stop_id"],
        to_stop=row["to_stop_id"],
        via_mode=row["via_mode"],
        walk_mins=row["avg_transfer_walk_mins"],
        peak_hour=row["peak_hour_only"])

print("CONNECTED_TO relationships created")

CONNECTED_TO relationships created


#Create Journey Relationship

In [ ]:
with driver.session() as session:

    for _, row in journey_segments.iterrows():

        session.run("""

        MATCH (j:Journey {
            journey_id:$journey_id
        })

        MATCH (r:Route {
            route_id:$route_id
        })

        MATCH (v:Vehicle {
            vehicle_id:$vehicle_id
        })

        MERGE (j)-[:USES_ROUTE]->(r)

        MERGE (j)-[:USES_VEHICLE]->(v)

        MERGE (v)-[:OPERATES_ON]->(r)

        """,

        journey_id=row["journey_id"],
        route_id=row["route_id"],
        vehicle_id=row["vehicle_id"])

print("Journey relationships created")

Journey relationships created


#Create Passenger and Journey Relationship

In [ ]:

query = """
MATCH (p:Passenger), (j:Journey)
WHERE p.passenger_id = j.passenger_id
MERGE (p)-[:MADE]->(j);
"""

with driver.session() as session:
    result = session.run(query)
    print("Relationships created")

print("Relationships created.")

Relationships created
Relationships created.


#Create Passenger and Feedback Relationship

In [ ]:

query = """
MATCH (p:Passenger)
MATCH (f:Feedback)
WHERE p.passenger_id = f.passenger_id
MERGE (p)-[:SUBMITTED]->(f)
"""

with driver.session() as session:
    result = session.run(query)
    print("Relationships created")

print("Relationships created.")

Relationships created
Relationships created.


#Feedback and Route Relationship

In [ ]:

query = """
MATCH (f:Feedback)
MATCH (r:Route)
WHERE f.route_id = r.route_id
MERGE (f)-[:ABOUT_ROUTE]->(r)
"""

with driver.session() as session:
    result = session.run(query)
    print("Relationships created")

print("Relationships created.")

Relationships created
Relationships created.


#Feedback and Stop Relationship

In [ ]:

query = """
MATCH (f:Feedback)
MATCH (s:Stop)
WHERE f.stop_id = s.stop_id
MERGE (f)-[:ABOUT_STOP]->(s)
"""

with driver.session() as session:
    result = session.run(query)
    print("Relationships created")

print("Relationships created.")

Relationships created
Relationships created.


#Feedback and Journey Relationship

In [ ]:
query = """
MATCH (f:Feedback)
MATCH (j:Journey)
WHERE f.journey_id = j.journey_id
MERGE (f)-[:ABOUT_JOURNEY]->(j)
"""

with driver.session() as session:
    result = session.run(query)
    print("Relationships created")

print("Relationships created.")

Relationships created
Relationships created.


#Verify Neo4j Connectivity

In [ ]:
driver = GraphDatabase.driver(
    "neo4j+s://be0a3f61.databases.neo4j.io",
    auth=("be0a3f61", "wcEdfl7sSqjEjtNsdGiLIIuWzqZ7m2Q3OwVz47a3N24")
)

driver.verify_connectivity()

print("Connected Successfully")

Connected Successfully


#Create Started-at Relationship

In [ ]:
with driver.session() as session:

    session.run("""
    MATCH (j:Journey),
          (s:Stop)
    WHERE j.destination_stop_id = s.stop_id
    MERGE (j)-[:ENDED_AT]->(s)
    """)

print("ENDED_AT relationships created")

ENDED_AT relationships created


#Relationship Created Summary

In [ ]:
with driver.session() as session:

    session.run("""
    MATCH (j:Journey),
          (s:Stop)

    WHERE j.origin_stop_id = s.stop_id

    MERGE (j)-[:STARTED_AT]->(s)
    """)

print("STARTED_AT relationships created")

STARTED_AT relationships created
